# Ember a hurokban: előzetes műveleti kapuk, kockázati szintek és audit naplózás

Ennek a leckének a README-je bevezeti az Ember a hurokban (Human-in-the-Loop) fogalmát egy rövid részlettel, amelyben a felhasználótól `APPROVE` vagy `REJECT` válasz kérése történik miután az ügynök már adott választ. Ez a minta jó kiindulópont, de a gyártási HITL megvalósítások gyakran három további elemet igényelnek:

1. Egy **előzetes műveleti kapu**, amely **még az ügynök általi kockázatos lépés végrehajtása előtt** fut, hogy a költség, visszafordíthatatlanság és késleltetés kontroll alatt maradjon.
2. **Kockázati szintű bontás**, így az alacsony kockázatú műveletek automatikusan végrehajtódnak, a közepes kockázatúak csoportosan jóváhagyhatók, és csak a magas kockázatú lépések várnak emberi beavatkozásra.
3. Egy **audit napló és felülvizsgálati kör**, ahol minden kapu döntés JSONL formátumban rögzítésre kerül, és egy elutasítás esetén az ügynök újrapromptolása struktúrált indokkal történik a „Revising...” egyszerű kiírása helyett.

Ez a jegyzetfüzet mindegyiket ugyanazon primitívek fölé építi, mint a `06-system-message-framework.ipynb`. Végigfuttatható `DEMO_MODE = True` módban (interaktív beviteli prompt nélkül) vagy valódi `input()` promptokkal, amikor `DEMO_MODE = False`. Megjegyzés: DEMO_MODE esetén a harmadik cél újrapróbálása szkriptelt, így a kör mechanikája teljes mértékben látható. Valódi felülvizsgálaton alapuló újbóli osztályozáshoz `DEMO_MODE = False` és egy operátor szükséges.

**Nem része a témának (más leckék kezelik):** hitelesítés és hozzáférés-szabályozás (06-os lecke README fenyegetés 2), eszközhívás köztes réteg (14-es lecke MAF mélyebb elemzés), többügynökös vitaminták.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## 1. modell: Előzetes jóváhagyási kapu

A README HITL részlete először meghívja az ügynököt, majd megkéri a felhasználót, hogy hagyja jóvá a kimenetet. Ez egy **művelet utáni** folyamat. Az ügynök már végrehajtotta a feladatot, tehát az LLM hívás költsége már megtörtént, és bármilyen mellékhatás (elküldött e-mail, írt adatbázis sor, közzétett hozzászólás) már megtörtént.

Egy **művelet előtti** folyamat beszúrja a kaput, mielőtt az ügynök végrehajtaná a kockázatos lépést. Az ügynök javasolja a műveletet, a kapu dönt a végrehajtásról, és csak jóváhagyás esetén következik be a mellékhatás.

| Nézőpont | Művelet utáni jóváhagyás (README részlet) | Művelet előtti kapu (ez a jegyzetfüzet) |
|---|---|---|
| Mikor fut a jóváhagyás? | Miután az ügynök előállította a kimenetet | Mielőtt bármilyen mellékhatás bekövetkezne |
| LLM költség elutasítás esetén | Már megfizetett | Csak a javaslatért fizetett, nem a műveletért |
| Visszafordíthatatlan mellékhatások | Lehetséges (a művelet már megtörtént) | Megelőzve |
| Auditálhatóság | A jóváhagyás egy kiírt üzenet | A jóváhagyás egy időbélyegzővel, művelettel és indoklással ellátott JSON rekord |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Minta 2: Kockázati szintek

Nem minden művelet igényel emberi jóváhagyást. Egy nyilvános API ellenőrzése csak olvasásra más téttel bír, mint egy ügyfél e-mail küldése. Mindkettőt azonos módon kezelni pazarló a kezelői figyelmet és lassítja az ügynököt.

Egy egyszerű, 3 szintű modell:

| Szint | Példák | Jóváhagyási folyamat |
|---|---|---|
| `alacsony` (csak olvasás) | Tudásbázis keresése, repülőjáratok keresése, nyilvános weboldal lekérése | Automatikusan végrehajtva, audit naplózással |
| `közepes` (olcsó módosítás) | Eredmény gyorsítótárazása, számláló növelése, emlékeztető beütemezése | Automatikusan végrehajtva, de napi ellenőrzéssel |
| `magas` (külső felé irányuló vagy visszafordíthatatlan) | E-mail küldése, kártya terhelése, nyilvános csatornán posztolás | Emberi jóváhagyásra várakozás |

Ez egy szintbesorolás. A termelési rendszerek gyakran használnak finomabb szinteket (pl. AWS IAM jogosultsági szintek, szerepekhez kötött hozzáférési szintek). Az alábbi 3 szintes verzió a legkisebb hasznos változat olyan ügynök számára, amely keveri az olvasási és mellékhatásos műveleteket.

Az alábbi osztályozó kulcsszó alapú heurisztikát használ, hogy a demo determinisztikus és olcsó maradjon. Egy termelési rendszerben ezt tanult osztályozóra vagy szabálymotorra cserélnéd.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## 3. minta: Audit napló és felülvizsgálati ciklus

A `print("Response approved.")` nem audit napló. A bizalom érdekében minden kapu döntést strukturált eseményként kell rögzíteni, amelyet később lekérdezhetsz, lejátszhatsz vagy egy incidens áttekintéséhez csatolhatsz.

Két elem:

1. **Csak hozzáfűzhető JSONL.** Egy sor döntésenként, időbélyeggel, művelettel, szinttel, döntéssel, indoklással. Könnyű keresni, később könnyű valós naplózó rendszerbe továbbítani.
2. **Felülvizsgálati ciklus elutasítás esetén.** Amikor a kapu `deny`-t ad vissza, az ügynök újra megkérdezi önmagát az elutasítás okával a kontextusban, hogy a következő javaslat elkerülhesse a problémát.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## További források

Számos más nyilvános projekt valósít meg ezeknek a HITL mintáknak különböző változatait. Hasonlítsa össze a megközelítéseket, hogy megtalálja, mi illik leginkább a stack-jéhez:

- **LangChain** emberi beavatkozást igénylő eszközcsomagok ([dokumentáció](https://python.langchain.com/docs/integrations/tools/human_tools)): beilleszthető eszközcsomagok, amelyek végrehajtást szüneteltetnek emberi inputra.
- **AutoGen** `UserProxyAgent` ([v0.2 dokumentáció](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ ezt átszervezte): egy agent szerepet használ kifejezetten az ember képviseletére többügynökös beszélgetésekben.
- **Microsoft Agent Framework (MAF)** függvényhívási middleware ([dokumentáció](https://learn.microsoft.com/agent-framework/)): middleware, amely minden eszköz-/függvényhívás körül fut, alkalmas jogosultsági és jóváhagyási folyamatok vezérlésére.

Minden projekt másként kezeli a három al-mintát: a LangChain eszközként csomagolja be őket, az AutoGen egy agent szerepet használ, a Microsoft Agent Framework pedig függvényhívási middleware-t. Olvasson el egy-két implementációt a végéig, mielőtt eldönti, melyik tervezési megoldást választja a saját agentje számára.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
